In [1]:
import numpy as np
import UQpy

from UQpy.run_model.RunModel import RunModel
from UQpy.distributions import Normal, Uniform, Lognormal # aqui pode ser que mude dependendo da distribuição que forem usar... mas ele tem várias muito boas
from UQpy.distributions.collection.JointIndependent import JointIndependent
from UQpy.sensitivity.SobolSensitivity import SobolSensitivity
from UQpy.sensitivity.PostProcess import *
from UQpy.run_model.model_execution.PythonModel import PythonModel
from UQpy.sensitivity.ChatterjeeSensitivity import ChatterjeeSensitivity

In [2]:
import carla

from carla import immune_response, immune_response_wrapper, params, y0
# só fazendo o import do modelo mesmo... aqui vcs podem fazer o import de vcs. Se tiverem dúvida em como trabalhar com um modelo externo, me mandem msg. 
# Também sempre há a possibilidade de vcs rodarem o modelo previamente... me procurem se quisem aprender isso

In [3]:
model = PythonModel(
    model_script="carla.py",
    model_object_name="immune_response_wrapper",
    var_names=[r'$\pi_v$', r'$k_{v1}$', r'$k_{v2}$', r'$\alpha_b$',
               r'$\beta_{ps}$', r'$\beta_{pl}$', r'$\beta_{bm}$',
               r'$\delta_{pl}$', r'$B_0$', r'$\pi_{b2}$'],
)
runmodel_obj = RunModel(model=model)

In [4]:
nominal = [6.182e-1, 4.82e-5, 7.48e-7, 6.0e0,
           6.72e-4, 5.61e-6, 1e-6,
           2.4e-4, 2.5e5, 1.27e-8]

dists = [Lognormal(s=0.25, scale=0.1*v) for v in nominal]
dist_object = JointIndependent(dists)

In [5]:
'''
Aqui optei por fazer um loop para mostrar como avaliar a convergência. Aqui nesse exemplo é simples, pq o modelo roda rápido e da pra fazer isso.
Mas no modelo de vcs pode ser que rodar assim seja inviável.
'''

n_init = 10000
n_plus = 10000
n_end  = 10000000+1

for n in range(n_init, n_end, n_plus):
    SA = SobolSensitivity(runmodel_obj, dist_object)
    SA.run(n_samples=n, n_bootstrap_samples=int(0.1*n))

    print(f"Para {n} amostras os resultados foram:")
    print("First order:", SA.first_order_indices)
    print("Total order:", SA.total_order_indices)

Para 10000 amostras os resultados foram:
First order: [[ 6.68771379e-01]
 [ 5.88090119e-02]
 [ 1.07713112e-01]
 [-1.67155385e-03]
 [ 3.29891334e-04]
 [-1.89293496e-03]
 [-1.87530697e-03]
 [-1.87497781e-03]
 [ 2.38743917e-03]
 [-1.87448956e-03]]
Total order: [[ 0.8107118 ]
 [ 0.13104322]
 [ 0.1426767 ]
 [-0.01233979]
 [-0.00490829]
 [-0.01363313]
 [-0.0135931 ]
 [-0.01359348]
 [-0.01376232]
 [-0.01359339]]


KeyboardInterrupt: 